# 3 · Router

Lecciones 5 y 3 del módulo 1: [Router](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239412-lesson-5-router) y [LangSmith Studio](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58238461-lesson-3-langsmith-studio). El notebook original es [`router.ipynb`](../../langchain-academy/module-1/router.ipynb).

En el notebook anterior el modelo podía pedir una herramienta, pero nadie la ejecutaba. Aquí se agregan dos piezas: un nodo que **ejecuta** las herramientas y una arista condicional que decide si ir a ese nodo o terminar. El modelo pasa a ser un **router**: según lo que se le pida, el grafo toma un camino u otro.

## El modelo

Estos apuntes usan un modelo local con [Ollama](https://ollama.com), gratis y sin claves: `qwen2.5:7b`, que sabe llamar herramientas. Antes de ejecutar, con Ollama abierto:

```
ollama pull qwen2.5:7b
```

`temperature=0` hace que el modelo elija siempre la opción más probable, así las respuestas casi no cambian de una ejecución a otra.

El curso usa `gpt-4o` de OpenAI. Para usarlo, cambia la celda de abajo por:

```python
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o", temperature=0)
```

El resto del notebook queda igual: al grafo no le importa qué modelo hay detrás.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:7b", temperature=0)

In [ ]:
def multiplicar(a: int, b: int) -> int:
    """Multiplica a por b.

    Args:
        a: primer entero
        b: segundo entero
    """
    return a * b


llm_con_herramientas = llm.bind_tools([multiplicar])

## El nodo que ejecuta herramientas

`ToolNode` recibe una lista de herramientas. Cuando se ejecuta, toma el último mensaje del estado, ejecuta cada llamada a herramienta que trae y devuelve un `ToolMessage` por cada una, con el resultado.

Se puede probar solo, sin grafo, con un estado que tenga una llamada a herramienta escrita a mano:

In [ ]:
from langchain_core.messages import AIMessage
from langgraph.prebuilt import ToolNode

nodo_herramientas = ToolNode([multiplicar])
pedido = AIMessage(content="", tool_calls=[{"name": "multiplicar", "args": {"a": 6, "b": 7}, "id": "llamada-1"}])

nodo_herramientas.invoke({"messages": [pedido]})

El `ToolMessage` trae el resultado, `42`, y el mismo `tool_call_id` que la llamada. Así el modelo sabe qué resultado corresponde a qué pedido cuando hay varios.

## La arista condicional

`tools_condition` mira el último mensaje: si trae llamadas a herramientas devuelve `"tools"`, y si no, `END`. Es tan simple que se puede escribir a mano:

In [ ]:
from langgraph.graph import END


def mi_tools_condition(estado):
    ultimo = estado["messages"][-1]
    if ultimo.tool_calls:
        return "tools"
    return END


print(mi_tools_condition({"messages": [pedido]}))
print(mi_tools_condition({"messages": [AIMessage(content="Hola, ¿en qué te ayudo?")]}))

Por eso el nodo que ejecuta herramientas se llama `"tools"`: es el nombre que devuelve `tools_condition`. El grafo de abajo usa la versión que trae LangGraph, que hace lo mismo que la de arriba.

## El grafo

In [ ]:
from IPython.display import Image, display
from langgraph.graph import MessagesState, START, StateGraph
from langgraph.prebuilt import tools_condition


def llamar_llm(estado: MessagesState):
    return {"messages": [llm_con_herramientas.invoke(estado["messages"])]}


constructor = StateGraph(MessagesState)
constructor.add_node("llamar_llm", llamar_llm)
constructor.add_node("tools", ToolNode([multiplicar]))
constructor.add_edge(START, "llamar_llm")
constructor.add_conditional_edges("llamar_llm", tools_condition)
constructor.add_edge("tools", END)
grafo = constructor.compile()

display(Image(grafo.get_graph().draw_mermaid_png()))

Desde `llamar_llm` salen dos caminos: a `tools`, si el modelo pidió una herramienta, o directo a `END`. Después de `tools`, el grafo termina.

In [ ]:
from langchain_core.messages import HumanMessage

resultado = grafo.invoke({"messages": [HumanMessage(content="Hola, ¿cuánto es 2 multiplicado por 2?")]})
for m in resultado["messages"]:
    m.pretty_print()

Ahora sí se ejecutó la herramienta: el último mensaje es un `ToolMessage` con el resultado. Con un saludo, el grafo toma el otro camino:

In [ ]:
resultado = grafo.invoke({"messages": [HumanMessage(content="¡Hola! ¿Cómo estás?")]})
for m in resultado["messages"]:
    m.pretty_print()

Queda un detalle: cuando se usa la herramienta, lo último que pasa es el resultado de `multiplicar`. Nadie le explica a la persona en una frase que 2 × 2 = 4. Para eso el resultado tiene que volver al modelo, y eso es el **agente** del notebook 4.

## LangSmith Studio

LangSmith Studio es una interfaz web para ver un grafo, ejecutarlo y revisar cada paso. Los grafos de este módulo también están como archivos en [`langchain-academy/module-1/studio/`](../../langchain-academy/module-1/studio): `simple.py`, `router.py` y `agent.py`, registrados en `langgraph.json`. Para abrirlos, con el entorno del curso instalado:

```
cd langchain-academy/module-1/studio
copy .env.example .env
langgraph dev
```

Después se entra a `https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024` (en macOS o Linux, `cp` en vez de `copy`). `langgraph dev` viene en el paquete `langgraph-cli[inmem]`, y Studio pide una cuenta gratuita de LangSmith. `simple.py` no usa modelo. `router.py` y `agent.py` usan `ChatOpenAI`: para correrlos con Ollama hay que cambiar el modelo igual que en la celda del principio.

## Resumen

- `ToolNode` ejecuta las llamadas a herramientas del último mensaje y devuelve un `ToolMessage` por cada una.
- `tools_condition` va a `"tools"` si el último mensaje pide herramientas y a `END` si no.
- Con esas dos piezas, el modelo elige el camino del grafo: es un router.

## Ejercicios

1. Pide «Multiplica 3 por 4, y también 5 por 6». ¿Cuántas llamadas a herramienta hizo el modelo en un solo mensaje? ¿Cuántos `ToolMessage` devolvió `ToolNode`?
2. Reemplaza `tools_condition` por `mi_tools_condition` en el grafo y comprueba que el resultado no cambia.
3. Cambia la arista `tools → END` por `tools → llamar_llm`. ¿Qué pasa ahora con la pregunta de 2 por 2? Con ese cambio se arma el agente del notebook 4.